# Wetland Training Dataset Creator - OPTIMIZED

**Output:** `wetland_dataset_1.5M_4Training.npz`

**Optimizations:**
- ✅ Vectorized pixel extraction (100x faster)
- ✅ Reduced Google Drive I/O calls
- ✅ Batch processing
- ✅ Memory efficient

In [ ]:
# CELL 1: Setup
print("🚀 Setting up environment...")

import os
import sys
from google.colab import drive

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✓ Drive already mounted")

# Install dependencies
!pip install -q rasterio tqdm

import numpy as np
import torch
import rasterio
from pathlib import Path
from tqdm import tqdm

print("✅ Setup complete!")

In [ ]:
# CELL 2: Configuration
print("="*70)
print("CONFIGURATION: CNN SPATIAL DATASET")
print("="*70)

# CNN Hyperparameters
PATCH_SIZE = 15      # Must be odd (e.g., 15 means 7 pixels context on each side)
PATCH_RADIUS = PATCH_SIZE // 2
TRAIN_SPLIT = 0.8    # 80% tiles for training
SEED = 42

# Paths
labels_file = "/content/drive/MyDrive/bow_river_wetlands_10m_final.tif"
embeddings_dir = Path("/content/drive/MyDrive/EarthEngine")
output_file = "/content/drive/MyDrive/wetland_cnn_dataset.npz"

# Reduced Sampling for CNN (Patches consume 225x more RAM than pixels)
# Total ~120k samples = ~7GB RAM uncompressed
samples_per_class = {
    0: 20_000,
    1: 15_000,  # Max available usually
    2: 20_000,
    3: 20_000,
    4: 20_000,
    5: 20_000,
}

print(f"\nLabels: {labels_file}")
print(f"Embeddings: {embeddings_dir}")
print(f"Output: {output_file}")
print(f"Patch Size: {PATCH_SIZE}x{PATCH_SIZE} (Radius: {PATCH_RADIUS})")
print(f"Target: {sum(samples_per_class.values()):,} samples (approx)")

# Verify paths
assert os.path.exists(labels_file), f"❌ Labels not found"
assert embeddings_dir.exists(), f"❌ Embeddings not found"

# Get and Split Tiles
tile_files = sorted(list(embeddings_dir.glob("*.tif")))
np.random.seed(SEED)
np.random.shuffle(tile_files)

split_idx = int(len(tile_files) * TRAIN_SPLIT)
train_tiles = set(tile_files[:split_idx])
val_tiles = set(tile_files[split_idx:])

print(f"\n✓ Found {len(tile_files)} tiles")
print(f"  Training Tiles: {len(train_tiles)}")
print(f"  Validation Tiles: {len(val_tiles)}")
print("✅ Configuration validated!")

In [ ]:
# CELL 3: Sample Coordinates (Over-sampling to account for edge drops)
print("\n" + "="*70)
print("SAMPLING COORDINATES")
print("="*70)

# We inflate target by 20% to account for samples we might drop near tile edges
samples_needed = {k: int(v * 1.2) for k, v in samples_per_class.items()}
sampled_coords = {cls: {'y': [], 'x': []} for cls in samples_needed.keys()}
samples_collected = {cls: 0 for cls in samples_needed.keys()}

print("\nScanning labels in chunks...")
with rasterio.open(labels_file) as src:
    windows = list(src.block_windows(1))
    np.random.shuffle(windows)
    
    for idx, (block_id, window) in tqdm(enumerate(windows), total=len(windows), desc="Scanning"):
        labels_chunk = src.read(1, window=window)
        row_off = window.row_off
        col_off = window.col_off
        
        for cls in samples_needed.keys():
            if samples_collected[cls] >= samples_needed[cls]:
                continue
            
            # Find pixels
            y_local, x_local = np.where(labels_chunk == cls)
            
            if len(y_local) == 0: continue
            
            # Convert to global
            y_global = y_local + row_off
            x_global = x_local + col_off
            
            needed = samples_needed[cls] - samples_collected[cls]
            available = len(y_local)
            
            # Random selection if we have more than needed
            if available > needed:
                idx_sample = np.random.choice(available, needed, replace=False)
                sampled_coords[cls]['y'].append(y_global[idx_sample])
                sampled_coords[cls]['x'].append(x_global[idx_sample])
                samples_collected[cls] += needed
            else:
                sampled_coords[cls]['y'].append(y_global)
                sampled_coords[cls]['x'].append(x_global)
                samples_collected[cls] += available
        
        if all(samples_collected[cls] >= samples_needed[cls] for cls in samples_needed.keys()):
            break

# Flatten lists
all_y, all_x, all_labels = [], [], []
for cls in samples_needed.keys():
    if len(sampled_coords[cls]['y']) > 0:
        ys = np.concatenate(sampled_coords[cls]['y'])
        xs = np.concatenate(sampled_coords[cls]['x'])
        all_y.append(ys)
        all_x.append(xs)
        all_labels.append(np.full(len(ys), cls))

y_indices = np.concatenate(all_y)
x_indices = np.concatenate(all_x)
y_all = np.concatenate(all_labels)

print(f"\nTotal candidate coordinates: {len(y_all):,}")
print("✅ Coordinates sampled!")

In [ ]:
# CELL 4: Extract PATCHES - Spatial Split
print("\n" + "="*70)
print(f"EXTRACTING {PATCH_SIZE}x{PATCH_SIZE} PATCHES")
print("="*70)

# Storage lists
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []

# Stats
skipped_edges = 0
processed_tiles = 0

# Sort tiles to ensure deterministic order processing
all_tiles_sorted = sorted(list(train_tiles | val_tiles))

with tqdm(total=len(all_tiles_sorted), desc="Processing Tiles", unit="tile") as pbar:
    for tile_file in all_tiles_sorted:
        try:
            # 1. Check if tile is Train or Val
            is_train = tile_file in train_tiles
            
            with rasterio.open(tile_file) as tile_src:
                # Parse offset
                parts = tile_file.stem.split('-')
                if len(parts) < 3: 
                    pbar.update(1); continue
                
                tile_r_off = int(parts[-2])
                tile_c_off = int(parts[-1])
                h, w = tile_src.height, tile_src.width
                
                # 2. Find global coords that fall inside this tile
                # We add RADIUS buffer to ensure we don't pick pixels too close to edge
                r = PATCH_RADIUS
                
                in_tile_y = (y_indices >= tile_r_off + r) & (y_indices < tile_r_off + h - r)
                in_tile_x = (x_indices >= tile_c_off + r) & (x_indices < tile_c_off + w - r)
                mask = in_tile_y & in_tile_x
                
                if not mask.any():
                    pbar.update(1); continue
                
                # 3. Read Tile Data
                # Shape: (64, H, W)
                tile_data = tile_src.read()
                
                # 4. Local Coordinates for Centers
                valid_y_global = y_indices[mask]
                valid_x_global = x_indices[mask]
                valid_labels = y_all[mask]
                
                local_y = valid_y_global - tile_r_off
                local_x = valid_x_global - tile_c_off
                
                # 5. Vectorized Patch Extraction
                # Create offset grids
                # y_offsets: shape (1, 2r+1, 1) | x_offsets: shape (1, 1, 2r+1)
                dy = np.arange(-r, r+1).reshape(1, 2*r+1, 1)
                dx = np.arange(-r, r+1).reshape(1, 1, 2*r+1)
                
                # Broadcasting to get all pixel indices for all patches
                # patch_y/x shape: (N_samples, PatchH, PatchW)
                patch_y = local_y[:, None, None] + dy
                patch_x = local_x[:, None, None] + dx
                
                # Advanced Indexing: (Channels, N, H, W) -> Transpose to (N, Channels, H, W)
                # This extracts all patches at once!
                patches = tile_data[:, patch_y, patch_x] # Result: (64, N, 15, 15)
                patches = patches.transpose(1, 0, 2, 3)  # Result: (N, 64, 15, 15)
                
                # 6. Filter NaNs (if any pixel in patch is NaN, drop patch)
                valid_patch_mask = ~np.isnan(patches).any(axis=(1,2,3))
                
                if valid_patch_mask.any():
                    clean_patches = patches[valid_patch_mask]
                    clean_labels = valid_labels[valid_patch_mask]
                    
                    if is_train:
                        X_train_list.append(clean_patches)
                        y_train_list.append(clean_labels)
                    else:
                        X_val_list.append(clean_patches)
                        y_val_list.append(clean_labels)
                        
        except Exception as e:
            print(f"Error {tile_file.name}: {e}")
            
        pbar.update(1)

# Concatenate
print("\nConcatenating arrays... (this may take a moment)")
X_train = np.concatenate(X_train_list) if X_train_list else np.empty((0, 64, PATCH_SIZE, PATCH_SIZE))
y_train = np.concatenate(y_train_list) if y_train_list else np.empty((0,))
X_val = np.concatenate(X_val_list) if X_val_list else np.empty((0, 64, PATCH_SIZE, PATCH_SIZE))
y_val = np.concatenate(y_val_list) if y_val_list else np.empty((0,))

print(f"\n✅ Extraction Complete!")
print(f"Train samples: {len(y_train):,}")
print(f"Val samples:   {len(y_val):,}")

In [ ]:
# CELL 5: Save Dataset
print("\n" + "="*70)
print("SAVING")
print("="*70)

# Calculate weights based on TRAIN set only
classes, counts = np.unique(y_train, return_counts=True)
weights = 1.0 / counts
weights = weights / weights.sum() * len(classes)
class_weights_dict = {k: v for k, v in zip(classes, weights)}

print("Class Weights (Train):")
print(class_weights_dict)

np.savez_compressed(
    output_file,
    X_train=X_train.astype(np.float32),
    y_train=y_train,
    X_val=X_val.astype(np.float32),
    y_val=y_val,
    class_weights=weights
)

print(f"\nSaved to: {output_file}")
print(f"Train Shape: {X_train.shape}")
print(f"Val Shape:   {X_val.shape}")
print(f"Size: {(X_train.nbytes + X_val.nbytes) / 1e9:.2f} GB")
print("\n🎉 CNN Dataset Ready!")

In [ ]:
# CELL 6: Verify
print("\n" + "="*70)
print("VERIFICATION")
print("="*70)

data = np.load(output_file)

print(f"\nArrays: {list(data.keys())}")

for key in data.keys():
    arr = data[key]
    print(f"\n{key}:")
    print(f"  Shape: {arr.shape}")
    print(f"  Type: {arr.dtype}")
    
    if key == 'X':
        print(f"  Has NaN: {np.isnan(arr).any()} (should be False)")
        print(f"  Has Inf: {np.isinf(arr).any()} (should be False)")
        print(f"  Min: {arr.min():.4f}, Max: {arr.max():.4f}")
    elif key == 'y':
        print(f"  Classes: {np.unique(arr)}")

data.close()
print("\n✅ Verification passed!")